# 02 — Understanding Jobs vs. Pipelines

> **DataMindAI | Ahmed**

## The most common source of confusion in Databricks

| | **Jobs** | **Pipelines** |
|--|----------|--------------|
| **Purpose** | Orchestration — control the *flow* | Data processing — transform data |
| **What it does** | Runs tasks in sequence/parallel | Moves data Bronze → Silver → Gold |
| **Think of it as** | The conductor | The musicians |
| **Can contain** | Notebooks, SQL, pipelines, JARs | Delta Live Tables (DLT) tables |
| **Databricks name** | Lakeflow Jobs | Lakeflow Pipelines |

### Key rule
> A **Job** can *orchestrate* multiple **Pipelines**.  
> A **Pipeline** cannot orchestrate a Job.

```
Lakeflow Job (orchestrator)
  │
  ├── Task 1: ingest_bronze      (Notebook task)
  ├── Task 2: bronze_pipeline    (DLT Pipeline task)
  ├── Task 3: silver_pipeline    (DLT Pipeline task)
  └── Task 4: gold_aggregation   (SQL task)
```

---
## Demo — Simulating a Bronze Pipeline (data processing)
---

In [ ]:
# ── This simulates what a Lakeflow PIPELINE does ─────────────────────────────
# A pipeline's only job is to process and transform data.
# It does not know about schedules, loops, or conditionals.

from pyspark.sql import Row
from pyspark.sql.functions import lit, current_timestamp

# Raw student records arrive as if from a source system
raw_data = [
    Row(student_id='S001', name='Aisha Rahman',    course='Data Science', score=82, attendance=95),
    Row(student_id='S002', name='James Bradley',   course='Data Science', score=41, attendance=60),
    Row(student_id='S003', name='Priya Patel',     course='Engineering',  score=76, attendance=88),
    Row(student_id='S004', name='Carlos Mendez',   course='Engineering',  score=55, attendance=72),
    Row(student_id='S005', name='Sophie Williams', course='Mathematics',  score=91, attendance=98),
    Row(student_id='S006', name='Kwame Asante',    course='Mathematics',  score=38, attendance=55),
    Row(student_id='S007', name='Liu Wei',         course='Data Science', score=67, attendance=80),
    Row(student_id='S008', name='Emma Johnson',    course='Engineering',  score=74, attendance=85),
    Row(student_id='S009', name='Tariq Ahmed',     course='Data Science', score=29, attendance=40),
    Row(student_id='S010', name='Fatima Al-Sayed', course='Mathematics',  score=88, attendance=94),
]

df_bronze = (spark.createDataFrame(raw_data)
               .withColumn('ingested_at', current_timestamp())
               .withColumn('source', lit('student_mis_system')))

df_bronze.write.format('delta').mode('overwrite').saveAsTable('demo_catalog.bronze.students_raw')
print(f'PIPELINE task: Bronze layer written — {df_bronze.count()} records')
df_bronze.printSchema()

---
## Demo — Simulating a Silver Pipeline (data processing)
---

In [ ]:
# ── Silver pipeline — clean and validate ─────────────────────────────────────
from pyspark.sql.functions import when, col, current_timestamp

df = spark.table('demo_catalog.bronze.students_raw')

# Data quality: filter out records missing student_id
df_valid = df.filter(col('student_id').isNotNull())

# Standardise course names to title case
df_silver = (df_valid
    .withColumn('course', when(col('course') == 'data science', 'Data Science')
                          .otherwise(col('course')))
    .withColumn('score_band',
        when(col('score') >= 70, 'High')
        .when(col('score') >= 50, 'Medium')
        .otherwise('Low'))
    .withColumn('processed_at', current_timestamp())
)

df_silver.write.format('delta').mode('overwrite').saveAsTable('demo_catalog.silver.students_clean')
print(f'PIPELINE task: Silver layer written — {df_silver.count()} records')
df_silver.select('student_id','name','course','score','score_band').show()

---
## Demo — What a JOB adds on top
---

### What the Job layer provides that pipelines cannot

The cells below simulate what the **Job orchestrator** does:
- Controls the order (Bronze first, then Silver, then Gold)
- Adds retry logic
- Sends notifications
- Branches based on conditions
- Loops over dynamic lists

The pipelines just do data work — the **Job** decides when and how they run.

In [ ]:
# ── Simulates the JOB orchestration layer ────────────────────────────────────
# In a real Lakeflow Job, each of these steps would be a separate TASK.
# The Job manages sequencing, failures, and notifications.

import datetime

def run_task(name, fn):
    start = datetime.datetime.now()
    print(f'[JOB] Starting task: {name}')
    try:
        fn()
        elapsed = (datetime.datetime.now() - start).seconds
        print(f'[JOB] Task SUCCEEDED: {name} ({elapsed}s)')
        return 'SUCCESS'
    except Exception as e:
        print(f'[JOB] Task FAILED: {name} — {e}')
        return 'FAILED'

# Simulate the DAG execution order
results = {}
results['ingest_bronze'] = run_task('ingest_bronze',
    lambda: print('  -> Bronze pipeline ran'))

if results['ingest_bronze'] == 'SUCCESS':
    results['process_silver'] = run_task('process_silver',
        lambda: print('  -> Silver pipeline ran'))

    if results['process_silver'] == 'SUCCESS':
        results['aggregate_gold'] = run_task('aggregate_gold',
            lambda: print('  -> Gold aggregation ran'))

print()
print('=== JOB RUN SUMMARY ===')
for task, status in results.items():
    print(f'  {task:25s} {status}')
print('========================')